# Inferencia por Lotes para la Predicción de Marketing Bancario

## 1. Importar Bibliotecas Necesarias

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import json

## 2. Cargar Modelo Entrenado

In [ ]:
model = None
model_path = '../training_pipeline/best_tuned_model.joblib'
feature_names_path = '../feature_pipeline/selected_feature_names.json'
trained_model_features = []

if os.path.exists(model_path):
    try:
        model = joblib.load(model_path)
        print(f"Modelo entrenado cargado exitosamente desde '{model_path}'.")
        
        # Cargar los nombres de las características con las que se entrenó el modelo
        if os.path.exists(feature_names_path):
            with open(feature_names_path, 'r') as f:
                trained_model_features = json.load(f)
            print(f"Nombres de características con las que se entrenó el modelo cargados desde '{feature_names_path}'. Características: {trained_model_features[:5]}... ({len(trained_model_features)} total)")
        elif hasattr(model, 'feature_names_in_'): # Alternativa para algunas versiones/modelos de sklearn
            trained_model_features = model.feature_names_in_.tolist()
            print(f"Nombres de características recuperados del atributo del modelo 'feature_names_in_'. Características: {trained_model_features[:5]}... ({len(trained_model_features)} total)")
        else:
            print(f"Advertencia: Archivo de nombres de características '{feature_names_path}' no encontrado, y el modelo no tiene 'feature_names_in_'. La consistencia del orden y nombres de las características será crítica y se asume.")
            
    except Exception as e:
        print(f"Error cargando modelo desde '{model_path}': {e}")
        model = None
else:
    print(f"Error: Archivo de modelo no encontrado en '{model_path}'. Asegúrate de que el pipeline de entrenamiento se haya ejecutado.")

## 3. Cargar Datos de Características para Inferencia

Para este ejercicio, estamos utilizando el archivo completo `bank-features-selected.csv` como un nuevo lote de datos para la inferencia. En un escenario del mundo real, estos serían datos nuevos y no vistos que han pasado por los mismos pasos de preprocesamiento que los datos de entrenamiento (manejados por el pipeline de características). El archivo `bank-features-selected.csv` idealmente debería contener solo las características con las que se entrenó el modelo y en el orden correcto.

In [ ]:
inference_data_path = '../feature_pipeline/bank-features-selected.csv'
X_inference = None
inference_df_for_output = None # Para almacenar datos para unir con predicciones

if os.path.exists(inference_data_path) and model is not None:
    try:
        inference_df = pd.read_csv(inference_data_path)
        print(f"Datos de inferencia cargados desde '{inference_data_path}'. Dimensiones: {inference_df.shape}")
        inference_df_for_output = inference_df.copy() # Guardar una copia para la salida
        
        # Si la columna objetivo 'y' está presente, eliminarla para X_inference
        if 'y' in inference_df.columns:
            X_inference = inference_df.drop(columns=['y'])
            print("Columna objetivo 'y' eliminada de los datos de inferencia para X_inference.")
        else:
            X_inference = inference_df.copy()
            print("Columna objetivo 'y' no encontrada en los datos de inferencia, usando todas las columnas como características para X_inference.")
        
        # Asegurar que los nombres y el orden de las características coincidan con los que el modelo fue entrenado
        if trained_model_features: # Si tenemos la lista de características con las que se entrenó el modelo
            missing_cols_in_data = set(trained_model_features) - set(X_inference.columns)
            extra_cols_in_data = set(X_inference.columns) - set(trained_model_features)
            
            if missing_cols_in_data:
                print(f"Error: Las siguientes características esperadas por el modelo FALTAN en los datos de inferencia: {missing_cols_in_data}")
                X_inference = None # Invalidar X_inference
            elif extra_cols_in_data:
                print(f"Advertencia: Las siguientes características en los datos de inferencia NO eran esperadas por el modelo y serán ELIMINADAS: {extra_cols_in_data}")
                X_inference = X_inference[trained_model_features] # Seleccionar solo características esperadas en el orden correcto
                print(f"Características para inferencia (X_inference) preparadas. Dimensiones: {X_inference.shape}")
            else: # Las columnas coinciden exactamente (o solo necesitan reordenarse)
                X_inference = X_inference[trained_model_features]
                print(f"Características para inferencia (X_inference) preparadas. Dimensiones: {X_inference.shape}")
                print("Columnas confirmadas/reordenadas para coincidir con el orden de características de entrenamiento del modelo.")
        elif X_inference is not None: 
            print(f"Advertencia: Los nombres de las características con las que se entrenó el modelo no se conocen definitivamente. Asumiendo que las columnas en '{inference_data_path}' son correctas y están en orden.")
            print(f"Características para inferencia (X_inference) preparadas. Dimensiones: {X_inference.shape}")
            
    except Exception as e:
        print(f"Error cargando o procesando datos de inferencia: {e}")
        X_inference = None
else:
    if model is None:
        print("Modelo no cargado. No se puede proceder con la carga de datos de inferencia.")
    else:
        print(f"Error: Archivo de datos de inferencia no encontrado en '{inference_data_path}'.")

## 4. Generar Predicciones

In [ ]:
predictions_labels = None
predictions_probabilities = None

if model is not None and X_inference is not None:
    print("Generando predicciones...")
    try:
        predictions_labels = model.predict(X_inference)
        print("Predicciones de etiquetas de clase generadas.")
        
        if hasattr(model, 'predict_proba'):
            predictions_probabilities = model.predict_proba(X_inference)[:, 1] # Probabilidad de la clase positiva (1)
            print("Probabilidades de predicción generadas para la clase positiva.")
        else:
            print("El modelo no soporta predict_proba(). Omitiendo predicciones de probabilidad.")
            
    except ValueError as ve:
        print(f"ValueError durante la predicción: {ve}")
        print("Esto ocurre a menudo si las características en los datos de inferencia no coinciden con las expectativas del modelo (nombres, orden, tipo o cantidad).")
        print(f"Características esperadas por el modelo (cantidad: {len(trained_model_features)}): {trained_model_features[:10]}...")
        print(f"Características reales proporcionadas (cantidad: {len(X_inference.columns)}): {X_inference.columns.tolist()[:10]}...")
        # X_inference.info()
    except Exception as e:
        print(f"Error general durante la predicción: {e}")
else:
    print("Modelo o datos de inferencia no disponibles/preparados. Omitiendo predicciones.")

## 5. Guardar Predicciones

In [ ]:
output_predictions_path = 'predictions.csv'

if predictions_labels is not None and inference_df_for_output is not None:
    print(f"Guardando predicciones en '{output_predictions_path}'...")
    try:
        # Crear un DataFrame para las predicciones. 
        # Usaremos 'inference_df_for_output', que es una copia de los datos de inferencia cargados originalmente.
        # Esto asegura que mantengamos todas las columnas originales para el contexto y añadamos columnas de predicción.
        # Es importante que 'predictions_labels' y 'predictions_probabilities' se alineen con 'inference_df_for_output'.
        # Esta alineación se maneja implícitamente si X_inference se derivó correctamente de inference_df y no se eliminaron filas de X_inference después de la carga.
        # Si X_inference tuviera filas eliminadas que no estaban en inference_df_for_output, esto daría error o se desalinearía.
        # Nuestra lógica actual para la creación de X_inference (eliminar 'y' y reordenar/seleccionar columnas basadas en trained_model_features)
        # debería mantener la integridad de las filas en relación con el inference_df cargado originalmente.

        results_df = inference_df_for_output.copy() # Comenzar con una copia de los datos originales cargados para inferencia
        results_df['predicted_label'] = predictions_labels
        
        if predictions_probabilities is not None:
            results_df['prediction_probability_yes'] = predictions_probabilities
            
        results_df.to_csv(output_predictions_path, index=False)
        print(f"Predicciones guardadas exitosamente en '{output_predictions_path}'.")
        print("\nPrimeras 5 filas del archivo de predicciones:")
        display(results_df.head())
        
    except Exception as e:
        print(f"Error guardando predicciones: {e}")
else:
    print("Predicciones o datos de inferencia originales para salida no disponibles. Omitiendo guardado de predicciones.")

### Contenido de `predictions.csv`

El archivo `predictions.csv` contiene los datos originales del lote de entrada (`bank-features-selected.csv`) junto con las siguientes nuevas columnas:

-   `predicted_label`: La etiqueta de clase predicha para cada instancia (0 para 'no', 1 para 'yes' - indicando si se predice que el cliente se suscribirá a un depósito a plazo).
-   `prediction_probability_yes` (opcional): Si el modelo admite predicciones de probabilidad y se generaron, esta columna contiene la probabilidad de que la instancia pertenezca a la clase positiva ('yes'). Esto proporciona una medida de confianza en la predicción.